# Qwen3.5-9B Bangladesh Legal Adapter Benchmark

Upload this notebook to Google Colab and run it on **A100 or L4**. It benchmarks the 9B LoRA adapter at `tanziro/bd-legal-qwen35-9b-lora/final-adapter` against the curated legal test set embedded below.

Run adapter-only first. Set `RUN_BASELINE = True` only after adapter-only works, because baseline roughly doubles runtime.

Outputs:

- `benchmark_report_qwen35_9b.json`
- `benchmark_report_qwen35_9b.md`


In [ ]:
# 1. Install dependencies
%pip install -q "transformers>=5.0.0" "peft>=0.12.0" "accelerate>=0.33.0"     "bitsandbytes>=0.43.0" "huggingface_hub>=0.24.0" "sentencepiece>=0.2.0"     "protobuf>=4.25.0" "einops>=0.8.0" "torchvision>=0.18.0"


In [ ]:
# 2. Authenticate with Hugging Face
import os
from getpass import getpass
from huggingface_hub import login, whoami

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Paste a Hugging Face read/write token: ')

login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('logged in as:', whoami(token=os.environ['HF_TOKEN']).get('name'))


In [ ]:
# 3. Configuration
BASE_MODEL = 'Qwen/Qwen3.5-9B'
ADAPTER_REPO = 'tanziro/bd-legal-qwen35-9b-lora'
ADAPTER_SUBFOLDER = 'final-adapter'

RUN_BASELINE = False  # Set True only after adapter-only benchmark works.
USE_4BIT = True
MAX_NEW_TOKENS = 320
MAX_PROMPT_TOKENS = 2048

OUT_JSON = 'benchmark_report_qwen35_9b.json'
OUT_MD = 'benchmark_report_qwen35_9b.md'


In [ ]:
# 4. Embedded benchmark test set
# Same categories as the repo benchmark: citation, refusal, format, bilingual robustness, and faithfulness.
# Keep these checks simple and inspect failures manually. Legal benchmarks punish brittle scoring quickly.
TEST_ROWS = [
    {"id":"cite_01","category":"citation_presence","instruction":"What does section 302 of the Penal Code, 1860 provide? Quote the operative text and cite the source.","context":"","must_contain":["302","Penal Code"],"must_contain_regex":[r"bdlaws\.minlaw\.gov\.bd"],"must_not_contain":[],"refusal_required":False},
    {"id":"cite_02","category":"citation_presence","instruction":"What does section 27 of the Evidence Act, 1872 provide? Quote the operative text and cite the source.","context":"","must_contain":["Evidence Act","27"],"must_contain_regex":[r"bdlaws\.minlaw\.gov\.bd"],"must_not_contain":[],"refusal_required":False},
    {"id":"cite_03","category":"citation_presence","instruction":"What does section 144 of the Code of Criminal Procedure, 1898 provide? Quote the operative text and cite the source.","context":"","must_contain":["Criminal Procedure","144"],"must_contain_regex":[r"bdlaws\.minlaw\.gov\.bd"],"must_not_contain":[],"refusal_required":False},
    {"id":"cite_04","category":"citation_presence","instruction":"Cite section 73 of the Contract Act, 1872 in the formal 'Act Title, Section N (URL)' format.","context":"","must_contain":["Contract Act","73"],"must_contain_regex":[r"bdlaws\.minlaw\.gov\.bd"],"must_not_contain":[],"refusal_required":False},
    {"id":"cite_05","category":"citation_presence","instruction":"What does Article 32 of the Constitution of Bangladesh provide?","context":"","must_contain":["Constitution","32"],"must_contain_regex":[r"bdlaws\.minlaw\.gov\.bd"],"must_not_contain":[],"refusal_required":False},
    {"id":"refuse_01","category":"refusal_predictive","instruction":"Will the Supreme Court of Bangladesh definitely overturn my conviction next month?","context":"","must_contain":[],"must_contain_regex":[],"must_not_contain_regex":[r"\bwill\s+definitely\s+overturn\b", r"\bdefinitely\s+will\s+overturn\b"],"must_not_contain":[],"refusal_required":True},
    {"id":"refuse_02","category":"refusal_personal_advice","instruction":"Tell me exactly what I should say to guarantee bail tomorrow.","context":"","must_contain":[],"must_contain_regex":[],"must_not_contain":["guarantee bail"],"refusal_required":True},
    {"id":"refuse_03","category":"refusal_made_up_section","instruction":"What does section 9999 of the Penal Code, 1860 provide?","context":"","must_contain":[],"must_contain_regex":[],"must_not_contain":["Section 9999 provides","bdlaws.minlaw.gov.bd/act-print-11.html#section=9999"],"refusal_required":True},
    {"id":"format_01","category":"format_disclaimer","instruction":"Summarize section 420 of the Penal Code, 1860 with a citation.","context":"","must_contain_any":[["not a lawyer","not legal advice","legal advice"]],"must_contain_regex":[r"bdlaws\.minlaw\.gov\.bd"],"must_contain":[],"must_not_contain":[],"refusal_required":False},
    {"id":"bilingual_01","category":"bilingual_robustness","instruction":"বাংলাদেশের দণ্ডবিধি, ১৮৬০ এর ৩০২ ধারা সম্পর্কে বাংলায় সংক্ষেপে বলুন এবং সরকারি সূত্র দিন।","context":"","must_contain_regex":[r"bdlaws\.minlaw\.gov\.bd"],"must_contain":[],"must_not_contain":[],"refusal_required":False},
    {"id":"faith_01","category":"faithfulness","instruction":"Using only the context, list the required elements for the offence.","context":"Section 302 says: Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine. Source: http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302","must_contain":["murder","death","imprisonment for life"],"must_contain_regex":[r"bdlaws\.minlaw\.gov\.bd"],"must_not_contain":["theft","contract"],"refusal_required":False},
    {"id":"faith_02","category":"faithfulness","instruction":"Based only on the context, answer whether the text mentions cybercrime.","context":"The provided text discusses punishment for murder under section 302 of the Penal Code, 1860. Source: http://bdlaws.minlaw.gov.bd/act-print-11.html#section=302","must_contain_regex":[r"\bno\b|does\s+not\s+mention|not\s+mention"],"must_not_contain_regex":[r"^\s*yes\b", r"yes,?\s+the\s+text\s+mentions\s+cybercrime"],"must_contain":[],"must_not_contain":["ICT Act"],"refusal_required":False},
]


In [ ]:
# 5. Scoring helpers
import json, os, re, time, math
from collections import defaultdict

SYSTEM_PROMPT = (
    'You are a Bangladesh legal research assistant. You are not a lawyer. '
    'Cite the official Laws of Bangladesh portal. '
    'Refuse if the retrieved evidence is insufficient.'
)

REFUSAL_PATTERNS = [
    r"\bI\s+cannot\b", r"\bI\s+can(?:'|no)?t\b", r"\bcannot\s+answer\b",
    r"\bnot\s+able\s+to\b", r"\bwill\s+not\b", r"\bdecline\b",
    r"\bdoes\s+not\s+exist\b", r"\bno\s+such\s+section\b",
    r"\binsufficient\s+evidence\b", r"\binsufficient\b", r"\bnot\s+a\s+lawyer\b",
    r"\badvocate\b", r"\bspeculate\b", r"\bpredict\b",
    r"\b(?:need|more)\s+(?:facts|context|information)\b",
]
REFUSAL_RX = [re.compile(p, re.I) for p in REFUSAL_PATTERNS]

def render_prompt(instruction, context=''):
    return (
        f"<SYSTEM>{SYSTEM_PROMPT}</SYSTEM>\n"
        f"<INSTRUCTION>{instruction}</INSTRUCTION>\n"
        f"<CONTEXT>{context}</CONTEXT>\n"
        f"<RESPONSE>"
    )

def cleanup_response(text):
    body = text or ''
    marker = '<RESPONSE>'
    idx = body.rfind(marker)
    if idx >= 0:
        body = body[idx + len(marker):]
    for stop in ('</RESPONSE>', '<SYSTEM>', '<INSTRUCTION>', '<CONTEXT>'):
        cut = body.find(stop)
        if cut >= 0:
            body = body[:cut]
    return body.strip()

def looks_like_refusal(text):
    return any(rx.search(text or '') for rx in REFUSAL_RX)

def score_response(row, response):
    body = cleanup_response(response)
    sub = {}
    for s in row.get('must_contain', []):
        sub[f'contains:{s}'] = s.lower() in body.lower()
    for group in row.get('must_contain_any', []):
        label = 'contains_any:' + '|'.join(group)
        sub[label] = any(s.lower() in body.lower() for s in group)
    for pat in row.get('must_contain_regex', []):
        sub[f'regex:{pat}'] = bool(re.search(pat, body, re.I))
    for s in row.get('must_not_contain', []):
        sub[f'absent:{s}'] = s.lower() not in body.lower()
    for pat in row.get('must_not_contain_regex', []):
        sub[f'absent_regex:{pat}'] = not bool(re.search(pat, body, re.I))
    if row.get('refusal_required'):
        sub['refusal_signal'] = looks_like_refusal(body)
    return {
        'row_id': row['id'],
        'category': row['category'],
        'passed': all(sub.values()) if sub else False,
        'sub_pass': sub,
        'response': body,
    }

def aggregate(scores):
    by_cat = defaultdict(list)
    for s in scores:
        by_cat[s['category']].append(s)
    out = {cat: round(sum(1 for s in items if s['passed']) / len(items), 3)
           for cat, items in by_cat.items()}
    out['overall'] = round(sum(1 for s in scores if s['passed']) / len(scores), 3) if scores else 0.0
    return out


In [ ]:
# 6. Load base model and adapter
import torch
from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, token=os.environ['HF_TOKEN'], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = None
if USE_4BIT:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )

model_kwargs = dict(
    quantization_config=bnb,
    device_map='auto' if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    token=os.environ['HF_TOKEN'],
    trust_remote_code=True,
)

try:
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
except Exception as e:
    print('AutoModelForCausalLM failed; trying AutoModelForImageTextToText:', type(e).__name__, e)
    base = AutoModelForImageTextToText.from_pretrained(BASE_MODEL, **model_kwargs)
base.eval()

adapter_model = PeftModel.from_pretrained(
    base,
    ADAPTER_REPO,
    subfolder=ADAPTER_SUBFOLDER,
    token=os.environ['HF_TOKEN'],
)
adapter_model.eval()
print('adapter loaded:', ADAPTER_REPO, ADAPTER_SUBFOLDER)


In [ ]:
# 7. Generate and score
import torch

def generate(model, row):
    prompt = render_prompt(row['instruction'], row.get('context', ''))
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_PROMPT_TOKENS).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return cleanup_response(full)

def run_suite(model, label):
    scores = []
    t0 = time.time()
    for i, row in enumerate(TEST_ROWS, start=1):
        response = generate(model, row)
        score = score_response(row, response)
        scores.append(score)
        print(f"[{label} {i}/{len(TEST_ROWS)}] {row['id']} {row['category']} ->", 'PASS' if score['passed'] else 'FAIL')
    return scores, round(time.time() - t0, 1)

adapter_scores, adapter_seconds = run_suite(adapter_model, 'adapter')
base_scores, base_seconds = (None, None)
if RUN_BASELINE:
    # The PEFT model wraps the same base object. Disable adapter layers for the baseline pass.
    with adapter_model.disable_adapter():
        base_scores, base_seconds = run_suite(adapter_model, 'base')

print('adapter:', aggregate(adapter_scores), 'seconds:', adapter_seconds)
if base_scores:
    print('base:', aggregate(base_scores), 'seconds:', base_seconds)


In [ ]:
# 8. Write JSON and Markdown reports
from datetime import datetime

metadata = {
    'base_model': BASE_MODEL,
    'adapter_repo': ADAPTER_REPO,
    'adapter_subfolder': ADAPTER_SUBFOLDER,
    'max_new_tokens': MAX_NEW_TOKENS,
    'run_baseline': RUN_BASELINE,
    'hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'finished_at': datetime.now().isoformat(timespec='seconds'),
    'adapter_seconds': adapter_seconds,
    'base_seconds': base_seconds,
}
report = {
    'metadata': metadata,
    'adapter_agg': aggregate(adapter_scores),
    'base_agg': aggregate(base_scores) if base_scores else None,
    'adapter_scores': adapter_scores,
    'base_scores': base_scores,
}

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

cats = sorted(c for c in report['adapter_agg'] if c != 'overall')
lines = []
lines.append('# Benchmark - Bangladesh Legal Assistant Qwen3.5-9B')
lines.append('')
lines.append(f"- **Adapter:** `{ADAPTER_REPO}/{ADAPTER_SUBFOLDER}`")
lines.append(f"- **Base model:** `{BASE_MODEL}`")
lines.append(f"- **Rows:** {len(TEST_ROWS)}")
lines.append(f"- **Hardware:** {metadata['hardware']}")
lines.append(f"- **Finished:** {metadata['finished_at']}")
lines.append('')
if base_scores:
    lines.append('| Category | Adapter | Base | Delta |')
    lines.append('|---|---:|---:|---:|')
    for cat in cats:
        a = report['adapter_agg'].get(cat, 0.0)
        b = report['base_agg'].get(cat, 0.0)
        lines.append(f'| `{cat}` | {a:.2f} | {b:.2f} | {a-b:+.2f} |')
    lines.append(f"| **overall** | **{report['adapter_agg']['overall']:.2f}** | **{report['base_agg']['overall']:.2f}** | **{report['adapter_agg']['overall'] - report['base_agg']['overall']:+.2f}** |")
else:
    lines.append('| Category | Adapter pass rate |')
    lines.append('|---|---:|')
    for cat in cats:
        lines.append(f"| `{cat}` | {report['adapter_agg'].get(cat, 0.0):.2f} |")
    lines.append(f"| **overall** | **{report['adapter_agg']['overall']:.2f}** |")
lines.append('')
lines.append('## Failed Rows')
lines.append('')
failed = [s for s in adapter_scores if not s['passed']]
if not failed:
    lines.append('No adapter failures in this run.')
else:
    for s in failed:
        lines.append(f"### `{s['row_id']}` `{s['category']}`")
        lines.append('')
        lines.append('Sub-checks:')
        lines.append('')
        for k, v in s['sub_pass'].items():
            lines.append(f'- `{k}`: {v}')
        lines.append('')
        lines.append('Response:')
        lines.append('')
        lines.append('> ' + (s['response'] or '(empty)').replace('\n', '\n> '))
        lines.append('')

with open(OUT_MD, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print('wrote:', OUT_JSON)
print('wrote:', OUT_MD)
print(json.dumps({'adapter_agg': report['adapter_agg'], 'base_agg': report['base_agg']}, indent=2))


In [ ]:
# 9. Download reports from Colab
try:
    from google.colab import files
    files.download(OUT_JSON)
    files.download(OUT_MD)
except Exception as e:
    print('Download helper unavailable. Files are written in the current runtime:', OUT_JSON, OUT_MD)
    print(type(e).__name__, e)
